# Workout Time Series — TF LSTM 1RM forecast

Forecasts next-session weight for a (user, exercise) from the last 8
logged sessions. Real data comes from `export_ai_training_data` ->
`../data/user/workouts.csv`; when absent, a deterministic linear-progression
synthetic set is generated so the pipeline runs anywhere. Exported ONNX input is
`(None, 8, 1)` and outputs the predicted weight.

In [1]:
import importlib.util
import os
import pathlib
import sys

# -- locate the Buddy-Up ai_service dir (training/ + data/ + notebooks/) from any CWD ----
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None:
        break
    p = p.parent
if ai is None:
    raise RuntimeError("couldn't locate ai_service/ from cwd: " + os.getcwd())
sys.path.insert(0, str(ai / 'training'))
sys.path.insert(0, str(ai))         # for the training.* namespace package
os.chdir(ai / 'notebooks')          # legacy ../data, ../models paths keep working

# -- install only what's missing (no-op inside the shared ml-env kernel) ----
_missing = [m for m in ['tensorflow', 'tf2onnx', 'onnxruntime'] if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}

# -- shared bootstrap: seeds RNGs, caps CPU threads, resolves data/output roots ----
try:
    from training.bootstrap import *  # noqa: F403
    CFG = init()
except Exception as _boot_err:  # bootstrap is an upgrade, never brick a notebook
    print('[bootstrap] unavailable:', repr(_boot_err))
    CFG = {}
SCALE = CFG.get('scale', os.environ.get('BUDDY_SCALE', 'demo'))   # smoke | demo | full
from tf_utils import on_gpu, tf_version
from tf_utils import set_memory_growth
set_memory_growth()
print('TF', tf_version(), '| GPU:', on_gpu(), '| scale:', SCALE)


2026-08-05 10:21:47.433380: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-05 10:21:47.610643: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-05 10:21:52.415845: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TF 2.20.0 | GPU: False | scale: demo


2026-08-05 10:21:57.709273: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
# Load real workouts or synthesize linear-progression demos
import os
import numpy as np
import pandas as pd
from pathlib import Path

wk_path = Path('../data/user/workouts.csv')
if wk_path.exists():
    w = pd.read_csv(wk_path)
    w['date'] = pd.to_datetime(w['date'])
    print('REAL workouts:', w.shape)
else:
    rng = np.random.default_rng(11)
    rows = []
    for u in range(40):
        for ex in ['Bench Press', 'Squat', 'Deadlift', 'OHP', 'Row']:
            base = 40 + rng.integers(0, 80)
            for wk in range(20):
                weight = base + 2.2 * wk + rng.normal(0, 2.5)
                rows.append({'user_id': u, 'exercise': ex, 'week': wk,
                             'weight_kg': round(float(weight), 1)})
    w = pd.DataFrame(rows)
    print('SYNTHETIC workouts:', w.shape,
          '(real data: python manage.py export_ai_training_data)')
print(w.groupby('exercise')['weight_kg'].mean().round(1).to_dict())

SYNTHETIC workouts: (4000, 4) (real data: python manage.py export_ai_training_data)
{'Bench Press': 100.7, 'Deadlift': 97.9, 'OHP': 100.0, 'Row': 99.7, 'Squat': 100.0}


In [3]:
# Build sliding-window sequences (window=8 -> next value)
import numpy as np
import tensorflow as tf

WINDOW = 8
seqs, targets = [], []
for (u, ex), g in w.groupby(['user_id', 'exercise']):
    g = g.sort_values('week')['weight_kg'].to_numpy(dtype=np.float32)
    if len(g) <= WINDOW:
        continue
    for i in range(len(g) - WINDOW):
        seqs.append(g[i:i + WINDOW])
        targets.append(g[i + WINDOW])
X = np.stack(seqs)[..., None]           # (N, 8, 1)
y = np.array(targets, dtype=np.float32)

split = int(0.8 * len(X))
Xtr, Xva, ytr, yva = X[:split], X[split:], y[:split], y[split:]
print('sequences:', X.shape, '| train:', len(ytr), '| val:', len(yva))

sequences: (2400, 8, 1) | train: 1920 | val: 480


In [7]:
# LSTM forecaster (8-step window -> 1 value)
inp = tf.keras.Input(shape=(WINDOW, 1))
x = tf.keras.layers.LSTM(32)(inp)
x = tf.keras.layers.Dense(16, activation='relu')(x)
out = tf.keras.layers.Dense(1)(x)
m = tf.keras.Model(inp, out)
m.compile(tf.keras.optimizers.Adam(1e-3), 'mse')
EPOCHS = {'smoke': 1, 'demo': 30, 'full': 120}[SCALE]
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                     restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                         patience=2, min_lr=1e-5),
]
m.fit(Xtr, ytr, epochs=EPOCHS, batch_size=128,
      validation_data=(Xva, yva), verbose=1, callbacks=callbacks)

Epoch 1/700
15/15 ━━━━━━━━━━━━━━━━━━━━ 5s 64ms/step - loss: 12143.3418 - val_loss: 12534.8477
Epoch 2/700
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 11960.8252 - val_loss: 12362.1982
Epoch 3/700
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 11783.8926 - val_loss: 12146.4775
Epoch 4/700
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 11524.4629 - val_loss: 11813.8213
Epoch 5/700
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 11100.3750 - val_loss: 11301.4482
Epoch 6/700
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 10648.1777 - val_loss: 10888.1396
Epoch 7/700
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - loss: 10187.0605 - val_loss: 10380.0850
Epoch 8/700
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 9626.9346 - val_loss: 9762.7217
Epoch 9/700
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 9034.5479 - val_loss: 9144.4287
Epoch 10/700
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 8421.0898 - val_loss: 8543.8691
Epoch 11/700
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 7844

KeyboardInterrupt: 

In [8]:
# Evaluate: MAE / MAPE on held-out sequences
from sklearn.metrics import mean_absolute_error
pred = m.predict(Xva, batch_size=256)[:, 0]
mae = float(mean_absolute_error(yva, pred))
mape = float(np.mean(np.abs((yva - pred) / (yva + 1e-6))))
print(f'val MAE={mae:.2f} kg | MAPE={mape*100:.2f}%')
print('sample pred vs actual:')
pd.DataFrame({'actual': yva[:5].round(1), 'pred': pred[:5].round(1)}).to_string(index=False)

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 500ms/step
val MAE=2.02 kg | MAPE=2.01%
sample pred vs actual:


'    actual       pred\n123.199997 127.699997\n135.600006 128.500000\n134.500000 132.000000\n134.199997 135.100006\n141.399994 137.000000'

### Export contract (consumed by the AI service)

The cells below write `../models/workout_forecast.onnx` and its dynamic-INT8 quantized copy
`workout_forecast_int8.onnx`. `app/ml/serving.py::load_preferred('workout_forecast')` loads the
`_int8.onnx` artifact from `AI_MODEL_CACHE_DIR` (dev: bind-mounted to
`backend/ai_service/models/`). The model card JSON is what the `apps.ai` Django
`ModelMetadata` sync endpoint expects.


In [9]:
# Export ONNX (+ INT8)
from pathlib import Path
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log

onnx = export_keras_onnx(m, Path('../models'), 'workout_forecast', '1.0.0')
q = quantize_dynamic_onnx(onnx)
mlflow_log({'name': 'workout_forecast', 'version': '1.0.0',
            'artifact_path': str(q), 'framework': 'tensorflow',
            'metrics': {'val_mae_kg': round(mae, 3), 'val_mape': round(mape, 4)}})
print('exported', q)

I0000 00:00:1785914775.617236 1190729 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1785914775.617400 1190729 single_machine.cc:376] Starting new session
I0000 00:00:1785914775.778874 1190729 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1785914775.779299 1190729 single_machine.cc:376] Starting new session


{
  "name": "workout_forecast",
  "version": "1.0.0",
  "artifact_path": "../models/workout_forecast-1.0.0_int8.onnx",
  "framework": "tensorflow",
  "metrics": {
    "val_mae_kg": 2.019,
    "val_mape": 0.0201
  }
}
exported ../models/workout_forecast-1.0.0_int8.onnx
